In [1]:
import pandas as pd
import numpy as np
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from google.oauth2.service_account import Credentials

In [ ]:
def connect_to_gsheet(creds_json,spreadsheet_name):
    scope = ["https://spreadsheets.google.com/feeds", 'https://www.googleapis.com/auth/spreadsheets',
             "https://www.googleapis.com/auth/drive.file", "https://www.googleapis.com/auth/drive"]
    
    credentials = ServiceAccountCredentials.from_json_keyfile_name(creds_json, scope)
    client = gspread.authorize(credentials)
    spreadsheet = client.open(spreadsheet_name)  # Access the first sheet
    return spreadsheet

In [33]:
n_interactions = 1000
n_users = 50

data = pd.DataFrame({
    "user_id": np.random.randint(1, n_users+1, n_interactions),
    "item_id": np.random.randint(1, n_items+1, n_interactions)
})

In [35]:
SPREADSHEET_NAME = 'Offline Data'
# SPREADSHEET_NAME_2 = 'Tab 2 Data'
# SHEET_NAME = 'Sheet1'
CREDENTIALS_FILE = './private_key.json'
sheet_by_name = connect_to_gsheet(CREDENTIALS_FILE, SPREADSHEET_NAME)

In [36]:
data_to_upload = data.values.tolist()
# data_to_upload
# sheet_by_name.clear()
ws = sheet_by_name.worksheet("interaction_data")
# ws.resize(rows=1, cols=1)  # Shrink sheet completely
# ws.clear()
ws.append_rows(data_to_upload)

{'spreadsheetId': '1p2HnhhrJCIs5cf8vZNroqxPe2Lbc25eblDl79wzvERA',
 'tableRange': 'interaction_data!A1:B1',
 'updates': {'spreadsheetId': '1p2HnhhrJCIs5cf8vZNroqxPe2Lbc25eblDl79wzvERA',
  'updatedRange': 'interaction_data!A2:B1001',
  'updatedRows': 1000,
  'updatedColumns': 2,
  'updatedCells': 2000}}

In [37]:
n_items = 100
categories = ['Electronics', 'Clothing', 'Books', 'Home', 'Toys']

item_table = pd.DataFrame({
    "item_id": range(1, n_items+1),
    "item_category": np.random.choice(categories, n_items),
    "item_price": np.round(np.random.uniform(5, 500, n_items),2),
    "item_rating": np.round(np.random.uniform(1, 5, n_items),1)
})

dataa = data.merge(item_table, on="item_id", how="left")

user_feat = dataa.groupby("user_id").agg(
    user_total_interactions=('item_id','count'),
    user_unique_items=('item_id','nunique'),
    user_unique_categories=('item_category','nunique'),
    user_avg_price=('item_price','mean'),
    user_avg_rating=('item_rating','mean'),
    user_price_std=('item_price','std'),
    user_rating_std=('item_rating','std')
).reset_index()

item_feat = dataa.groupby("item_id").agg(
    item_total_interactions=('user_id','count'),
    item_unique_users=('user_id','nunique'),
    item_avg_price=('item_price','mean'),
    item_avg_rating=('item_rating','mean'),
    item_price_std=('item_price','std'),
    item_rating_std=('item_rating','std')
).reset_index()

item_feat = item_feat.merge(item_table, on="item_id", how="left")

In [38]:
data_to_upload = user_feat.values.tolist()
# data_to_upload
# sheet_by_name.clear()
ws = sheet_by_name.worksheet("user_data")
# ws.resize(rows=1, cols=1)  # Shrink sheet completely
# ws.clear()
ws.append_rows(data_to_upload)

data_to_upload = item_feat.values.tolist()
# data_to_upload
# sheet_by_name.clear()
ws = sheet_by_name.worksheet("item_data")
# ws.resize(rows=1, cols=1)  # Shrink sheet completely
# ws.clear()
ws.append_rows(data_to_upload)

{'spreadsheetId': '1p2HnhhrJCIs5cf8vZNroqxPe2Lbc25eblDl79wzvERA',
 'tableRange': 'item_data!A1:J1',
 'updates': {'spreadsheetId': '1p2HnhhrJCIs5cf8vZNroqxPe2Lbc25eblDl79wzvERA',
  'updatedRange': 'item_data!A2:J51',
  'updatedRows': 50,
  'updatedColumns': 10,
  'updatedCells': 500}}